# Normal + Irradiance Pipeline for Wallpaper Visualization

This notebook runs the streamlined RGB→X variant that outputs only **normal maps** and **irradiance (shadow)** layers.

**Use case:** Extract shadow patterns from room photos to apply realistic lighting to warped wallpaper patterns.

**Make sure to enable GPU:** Runtime → Change runtime type → GPU (T4, V100, or A100)

## Setup: Clone Repository and Install Dependencies

**First-time setup:** You'll need a GitHub Personal Access Token to clone the private repo.

### How to generate token:
1. Go to: https://github.com/settings/tokens
2. Click **"Generate new token (classic)"**
3. Name it: `Colab RGB-X`
4. Check: **`repo`** (Full control of private repositories)
5. Click **"Generate token"** at bottom
6. **Copy the token** (you won't see it again!)

Paste the token when prompted below ⬇️

In [ ]:
# Clone YOUR private repo
# This requires a Personal Access Token for private repo access

from getpass import getpass
import os

print("🔐 GitHub Authentication Required")
print("Generate token at: https://github.com/settings/tokens")
print("Permissions needed: 'repo' (full control)")
print()

token = getpass('Enter your GitHub Personal Access Token: ')

# Clone using token
!git clone https://{token}@github.com/id-21/rgb-x.git

# Clear token from memory for security
token = None

# Navigate to the branch with normal_irradiance module
%cd rgb-x
!git checkout normal-irradiance-pipeline

## Install Python 3.9 Environment

**CRITICAL:** This code was built for Python 3.9. Colab defaults to Python 3.12, which causes compatibility issues with older dependencies (diffusers 0.20.0, JAX, etc.).

We create an isolated Python 3.9 virtual environment with exact package versions from `environment.yml`.

In [ ]:
# Force Python 3.9 environment (matching original environment.yml)
!apt-get update -qq
!apt-get install -y python3.9 python3.9-venv python3.9-dev

# Create and activate Python 3.9 virtual environment
!python3.9 -m venv /content/py39env
!source /content/py39env/bin/activate && pip install --upgrade pip

# Install PyTorch with CUDA 12.1 support (in Python 3.9 environment)
!source /content/py39env/bin/activate && pip install torch==2.3.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install exact versions from environment.yml
!source /content/py39env/bin/activate && pip install diffusers==0.20.0 gradio==3.48.0 imageio==2.34.1 numpy==1.26.4 opencv-python==4.9.0.80 transformers==4.40.2 huggingface-hub==0.25.0

print("\n✓ Python 3.9 environment created with all dependencies")

## Verify GPU Availability

In [ ]:
# Verify GPU is available (using Python 3.9 environment)
!/content/py39env/bin/python -c "import torch; print(f'CUDA available: {torch.cuda.is_available()}'); print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"None\"}');"

## Fix Colab Environment Issues

Colab sets environment variables for Jupyter notebooks that conflict with standalone Python scripts:

In [ ]:
import os

# Fix: Clear matplotlib backend issue (Colab sets this for Jupyter, conflicts with standalone Python)
os.environ.pop('MPLBACKEND', None)

print("✓ Environment fixes applied. Ready to launch demo!")

## Launch Normal + Irradiance Demo

This will generate a **public Gradio URL** (https://xxx.gradio.live) that you can access in your browser.

### What you'll get:
- **Normal Map:** Surface orientation in camera space
- **Irradiance (Shadow) Layer:** Lighting patterns including shadows

### Downloads:
- **Normal Map (NPY):** Raw numpy array in linear space for programmatic use
- **Normal Map (PNG):** Visual preview of surface normals
- **Shadow Layer (EXR):** Linear RGB format for ST-RoomNet V3 integration
- **Irradiance Preview (PNG):** Visual verification of shadow extraction

**Performance:** ~3x faster than full 5-AOV pipeline (2 inference calls vs 5)

In [ ]:
# Launch Normal + Irradiance Gradio demo (using Python 3.9)
%cd normal_irradiance
!/content/py39env/bin/python gradio_demo_normal_irradiance.py

## Using the Shadow EXR with ST-RoomNet V3

After downloading `irradiance_shadow.exr` from the demo above, use this code to apply shadows to your wallpaper:

```python
import cv2
import numpy as np

# 1. Load the shadow layer (linear RGB)
irradiance = cv2.imread('irradiance_shadow.exr', cv2.IMREAD_UNCHANGED)
irradiance = cv2.cvtColor(irradiance, cv2.COLOR_BGR2RGB)

# 2. Mask wall pixels (from ST-RoomNet normals)
wall_mask = (normals == target_wall_id)  # Your wall detection logic

# 3. Compute baseline brightness (98th percentile)
baseline = np.percentile(irradiance[wall_mask], 98)

# 4. Create relative shadow mask
shadow_mask = irradiance / baseline
shadow_mask = np.clip(shadow_mask, 0.4, 1.0)  # Preserve ambient light

# 5. Apply to wallpaper (in linear space!)
wallpaper_linear = wallpaper_srgb ** 2.2  # Convert to linear
shadowed = wallpaper_linear * shadow_mask  # Multiply
result_srgb = shadowed ** (1/2.2)  # Convert back to sRGB
```

**Key:** All multiplication must happen in **linear color space** to avoid hue shifts in saturated colors.

See `normal_irradiance/CLAUDE.md` and `research/understanding-shadow.md` for detailed explanations.

## Upload Your Own Images

Use the file upload widget below to upload room photos for testing:

In [ ]:
from google.colab import files
uploaded = files.upload()

# Your uploaded files will be in the current directory
for filename in uploaded.keys():
    print(f'Uploaded: {filename}')

## Troubleshooting

### Common Issues:

**1. "CUDA out of memory"**
- Input images are auto-resized to max 1000px
- If still failing, try restarting runtime and clearing RAM

**2. "Module not found" errors**
- Verify you're on the `normal-irradiance-pipeline` branch
- Re-run the Python 3.9 installation cell

**3. Gradio link not accessible**
- The demo already has `share=True` enabled
- If link expires, rerun the launch cell to get a new URL

**4. Wrong Python version**
- Always use `/content/py39env/bin/python` to run scripts
- System `python` or `python3` will use Colab's Python 3.12

See `implementation.md` for detailed error explanations and solutions.